# Collector Missing Data Spots (Polars)

Lists collector polling gaps and estimates missed polls and rows using the Polars analysis path.

In [1]:
from pathlib import Path
import importlib.util
import sys

import plotly.express as px
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "pyproject.toml").exists() and (candidate / "analysis" / "polars").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from the project root or a subdirectory.")

POLARS_ANALYSIS_DIR = PROJECT_ROOT / "analysis" / "polars"
polars_analysis_path = str(POLARS_ANALYSIS_DIR)
if polars_analysis_path in sys.path:
    sys.path.remove(polars_analysis_path)
sys.path.insert(0, polars_analysis_path)

for module_name in ("_shared", "report_cache", "cli_common"):
    module = sys.modules.get(module_name)
    module_file = getattr(module, "__file__", None) if module else None
    module_path = Path(module_file).resolve() if module_file else None
    if module_path is not None and POLARS_ANALYSIS_DIR not in module_path.parents:
        sys.modules.pop(module_name, None)

def load_polars_script(module_name: str, filename: str):
    spec = importlib.util.spec_from_file_location(module_name, POLARS_ANALYSIS_DIR / filename)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

missing_data = load_polars_script("polars_collector_missing_data_spots", "collector-missing-data-spots.py")

DB = PROJECT_ROOT / "data" / "foli.db"
CACHE_DIR = PROJECT_ROOT / "outputs" / "polars-report-cache"
FORCE_CACHE = False
NO_CACHE = False
SOURCE = None
LIMIT = 20
MIN_OBSERVATIONS = 1
GAP_MULTIPLIER = 2.0
MIN_MISSING_MINUTES = 0.0


In [2]:
settings = missing_data.ReportSettings(
    db=DB,
    limit=LIMIT,
    min_observations=MIN_OBSERVATIONS,
).resolved()
polls = missing_data.load_collector_polls(settings, source=SOURCE)
spots = missing_data.build_missing_spots(
    polls,
    gap_multiplier=GAP_MULTIPLIER,
    min_missing_minutes=MIN_MISSING_MINUTES,
)
summary = missing_data.summarize_missing_spots(spots, polls)
summary


source,poll_count,success_count,failed_count,missing_spot_count,total_missing_min,largest_missing_min,estimated_missed_polls,estimated_missed_rows
str,i64,i64,i64,i64,f64,f64,f64,f64
"""siri_vm""",42556,42454,102,25,216.23,47.77,432.42,57521.45
"""siri_alerts""",4310,4297,13,3,55.44,34.15,11.08,144.57
"""gtfs""",3,3,0,0,0.0,0.0,0.0,0.0


In [3]:
spots.head(LIMIT)


source,gap_start_utc,gap_end_utc,gap_min,expected_cadence_seconds,missing_min,estimated_missed_polls,estimated_missed_rows,failed_attempt_count,next_success_status
str,str,str,f64,f64,f64,f64,f64,i64,str
"""siri_vm""","""2026-04-28T02:40:08+00:00""","""2026-04-28T03:28:24+00:00""",48.27,30.0,47.77,95.53,12707.66,13,"""OK"""
"""siri_vm""","""2026-05-08T08:58:16+00:00""","""2026-05-08T09:34:17+00:00""",36.02,30.0,35.52,71.03,9448.72,0,"""OK"""
"""siri_alerts""","""2026-05-08T08:55:20+00:00""","""2026-05-08T09:34:29+00:00""",39.15,300.0,34.15,6.83,89.06,0,"""OK"""
"""siri_vm""","""2026-04-24T10:36:16+00:00""","""2026-04-24T10:55:12+00:00""",18.93,30.0,18.43,36.87,4903.93,7,"""OK"""
"""siri_vm""","""2026-05-07T07:16:36+00:00""","""2026-05-07T07:30:32+00:00""",13.93,30.0,13.43,26.87,3573.75,6,"""OK"""
…,…,…,…,…,…,…,…,…,…
"""siri_vm""","""2026-05-04T00:00:11+00:00""","""2026-05-04T00:08:26+00:00""",8.25,30.0,7.75,15.5,2061.78,5,"""OK"""
"""siri_vm""","""2026-05-01T00:00:27+00:00""","""2026-05-01T00:05:10+00:00""",4.72,30.0,4.22,8.43,1121.79,4,"""OK"""
"""siri_vm""","""2026-05-07T23:59:59+00:00""","""2026-05-08T00:04:42+00:00""",4.72,30.0,4.22,8.43,1121.79,4,"""OK"""


In [4]:
if summary.is_empty():
    print("No collector polls found.")
else:
    fig = px.bar(
        summary.sort("total_missing_min"),
        x="total_missing_min",
        y="source",
        orientation="h",
        title="Estimated missing-data duration by collector source",
        labels={"source": "Source", "total_missing_min": "Estimated missing minutes"},
    )
    fig.update_layout(showlegend=False)
    fig.show()
